# Support ticket tagging

**Manahil Iftikhar · DevelopersHub internship portfolio**

Zero-shot/few-shot prompts and controlled labels

**Execution status:** Prompt/parser checks pass offline; model quality not established

## Data and evidence

Three handcrafted support-ticket examples and five candidate labels; google/flan-t5-small. These examples are a demonstration, not a benchmark dataset.

The saved original output labels all three examples Login Problem. The original few-shot prompt is defined but never used. No calibrated probabilities or verified top-three ranking are present.

## Maintained workflow

The maintained code actually applies the selected prompting mode, accepts only known labels, removes duplicates, and flags invalid output for human review. Compare prompting modes on a separate labelled dataset before claiming accuracy.

Original assignment title: *Auto Tagging Support Tickets Using LLM*. Original code and outputs are preserved in `archive/`.

Install the environment described in the repository README, then select it as the VS Code notebook kernel. Outputs below are intentionally cleared until this maintained version is executed.

In [ ]:
from pathlib import Path
import sys

# Supports opening the notebook from the repository root or notebooks/ in VS Code.
ROOT = Path.cwd()
if not (ROOT / 'portfolio').is_dir():
    ROOT = ROOT.parent
if not (ROOT / 'portfolio').is_dir():
    raise RuntimeError('Open this notebook inside the cloned repository.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
DATA = ROOT / 'data'
ARTIFACTS = ROOT / 'artifacts'
ARTIFACTS.mkdir(exist_ok=True)

## Load the checkpoint

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from portfolio.tickets import tag_ticket

name = 'google/flan-t5-small'
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModelForSeq2SeqLM.from_pretrained(name)

## Compare zero-shot and few-shot prompts on demonstration inputs

In [ ]:
import pandas as pd

tickets = [
    'My internet connection is very slow since yesterday.',
    'I forgot my password and cannot log in to my account.',
    'The billing amount on my last invoice is incorrect.',
]
rows = []
for text in tickets:
    for mode in [False, True]:
        result = tag_ticket(text, tokenizer, model, few_shot=mode)
        rows.append({'ticket': text, 'mode': 'few-shot' if mode else 'zero-shot', **result})
pd.DataFrame(rows)

## Evaluate before claiming accuracy

These three examples are demonstrations. Create a separate labelled dataset, define whether multiple tags are allowed, and evaluate per-label precision/recall and invalid-output rate. Generated text is not a calibrated probability ranking; invalid labels require review.